# 02 Schema Audit

notebook นี้ตรวจไฟล์จริงทั้งหมดที่อยู่ใน `data/raw/`
ผลลัพธ์ใช้ยืนยัน column, จำนวนแถว, data type, missing values และ duplicate rows ก่อนเริ่ม Phase 3

In [1]:
from pathlib import Path

import pandas as pd
from pandas.api.types import is_numeric_dtype

## ค้นหา root ของโครงการ

รองรับการเปิด notebook จาก `notebooks/` หรือจาก root ของโครงการ โดยตรวจหา `data/raw/` ที่มีอยู่จริง

In [2]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path]
    candidate_paths.extend(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "data" / "raw").exists():
            return candidate_path

    raise FileNotFoundError("ไม่พบ root ของโครงการที่มี data/raw")


PROJECT_ROOT = find_project_root()

## กำหนด path หลัก

cell นี้กำหนดตำแหน่งของไฟล์ข้อมูลดิบจาก root ที่ตรวจพบ

In [3]:
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

## รวบรวมไฟล์ข้อมูลที่รองรับ

รองรับ CSV, XLSX, XLS และ JSON ตามไฟล์ที่ได้มาจริงใน Phase 2

In [4]:
SUPPORTED_SUFFIXES = {".csv", ".xlsx", ".xls", ".json"}
raw_files = sorted(DATA_RAW_DIR.glob("*"))
supported_files = []

for raw_file in raw_files:
    if raw_file.is_file() and raw_file.suffix.lower() in SUPPORTED_SUFFIXES:
        supported_files.append(raw_file)

## แสดงบัญชีรายชื่อไฟล์

ตารางนี้ยืนยันไฟล์ที่ถูกนำมา audit ในรอบการรันนี้

In [5]:
file_inventory_records = []

for raw_file in supported_files:
    file_inventory_records.append(
        {
            "file_name": raw_file.name,
            "file_path": str(raw_file.relative_to(PROJECT_ROOT)),
            "file_format": raw_file.suffix.lower().replace(".", "").upper(),
            "file_size_bytes": raw_file.stat().st_size,
        }
    )

file_inventory_df = pd.DataFrame(file_inventory_records)
file_inventory_df

,file_name,file_path,file_format,file_size_bytes
0,labai_land_matching.xlsx,data\raw\labai_land_matching.xlsx,XLSX,11346
1,labai_registered_land_demand_supply.json,data\raw\labai_registered_land_demand_supply.json,JSON,20871
2,labai_registered_landowners.json,data\raw\labai_registered_landowners.json,JSON,11433
3,sacit_craft_community_2564_2568.xlsx,data\raw\sacit_craft_community_2564_2568.xlsx,XLSX,17472
4,sacit_craft_product_overview_2568.csv,data\raw\sacit_craft_product_overview_2568.csv,CSV,4235


## อ่านไฟล์ CSV ด้วย encoding ที่ตรวจได้

ฟังก์ชันนี้ลอง encoding ที่ใช้ทั่วไปกับข้อมูลภาษาไทย และบันทึก encoding ที่อ่านสำเร็จ

In [6]:
def load_csv_data(file_path):
    encoding_candidates = ["utf-8-sig", "utf-8", "cp874"]

    for encoding_name in encoding_candidates:
        try:
            loaded_df = pd.read_csv(file_path, encoding=encoding_name)
            return loaded_df, encoding_name
        except UnicodeDecodeError:
            continue

    raise UnicodeError("ไม่สามารถอ่าน CSV ด้วย encoding ที่กำหนด")

## อ่านไฟล์ข้อมูลตามรูปแบบ

ฟังก์ชันนี้คืนตารางข้อมูลจริง โดยแยก sheet ของ Excel ออกเป็นคนละรายการ audit

In [7]:
def load_data_tables(file_path):
    file_suffix = file_path.suffix.lower()

    if file_suffix == ".csv":
        loaded_df, encoding_name = load_csv_data(file_path)
        return [("CSV", loaded_df, encoding_name)]

    if file_suffix in {".xlsx", ".xls"}:
        excel_file = pd.ExcelFile(file_path)
        loaded_tables = []

        for sheet_name in excel_file.sheet_names:
            loaded_df = pd.read_excel(file_path, sheet_name=sheet_name)
            loaded_tables.append((sheet_name, loaded_df, "not_applicable"))

        return loaded_tables

    if file_suffix == ".json":
        loaded_df = pd.read_json(file_path)
        return [("JSON", loaded_df, "utf-8")]

    raise ValueError(f"Unsupported file type: {file_suffix}")

## ระบุ column ที่อาจใช้เป็นมิติการวิเคราะห์

การระบุนี้อาศัยชื่อ column ที่พบจริงและใช้เพื่อคัดกรองเบื้องต้นเท่านั้น
ยังไม่แปลงข้อมูลหรือสรุปความหมายเชิงธุรกิจของ column

In [8]:
def find_columns_by_keywords(column_names, keywords):
    matched_columns = []

    for column_name in column_names:
        normalized_name = str(column_name).lower()

        for keyword in keywords:
            if keyword in normalized_name:
                matched_columns.append(str(column_name))
                break

    return matched_columns

## ระบุ column ตัวเลขที่อ่านได้หรือแปลงได้

ผลลัพธ์แยก column ที่เป็นตัวเลขตาม data type ออกจาก column ข้อความที่แปลงเป็นตัวเลขได้เกือบทั้งหมด

In [9]:
def identify_numeric_columns(dataframe):
    numeric_columns = []
    convertible_columns = []

    for column_name in dataframe.columns:
        series = dataframe[column_name]

        if is_numeric_dtype(series):
            numeric_columns.append(str(column_name))
            continue

        normalized_series = series.astype("string").str.replace(",", "", regex=False).str.strip()
        normalized_series = normalized_series.replace({"": pd.NA, "-": pd.NA})
        converted_series = pd.to_numeric(normalized_series, errors="coerce")
        non_missing_count = normalized_series.notna().sum()

        if non_missing_count == 0:
            continue

        conversion_ratio = converted_series.notna().sum() / non_missing_count

        if conversion_ratio >= 0.8:
            convertible_columns.append(str(column_name))

    return numeric_columns, convertible_columns

## สรุปวิธีวิเคราะห์ที่อาจทำได้

ข้อความนี้สรุปจากชนิดของ column ที่พบจริง ยังไม่ใช่การเลือก model หรือการรับรองว่าใช้ได้ทันที

In [10]:
def summarize_possible_analytics(date_columns, province_columns, category_columns, numeric_columns):
    possible_methods = ["descriptive summary"]
    usable_numeric_columns = numeric_columns

    if province_columns and usable_numeric_columns:
        possible_methods.append("provincial comparison")

    if category_columns and usable_numeric_columns:
        possible_methods.append("category comparison")

    if date_columns and usable_numeric_columns:
        possible_methods.append("time trend after date validation")

    if len(usable_numeric_columns) >= 2:
        possible_methods.append("numeric relationship inspection")

    if len(usable_numeric_columns) >= 2 and len(province_columns) > 0:
        possible_methods.append("clustering feasibility review")

    return ", ".join(possible_methods)

## ตรวจ schema และคุณภาพข้อมูลของทุกตาราง

cell นี้คำนวณจำนวนแถว จำนวน column, data type, missing values และ duplicate rows จากข้อมูลจริงทั้งหมด

In [11]:
schema_records = []
column_profile_records = []
load_error_records = []

date_keywords = ["date", "time", "วันที่", "ปี", "เดือน", "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
province_keywords = ["province", "จังหวัด"]
category_keywords = ["category", "ประเภท", "หมวด", "รายการสินค้า", "ชื่อชุมชน"]

for raw_file in supported_files:
    try:
        loaded_tables = load_data_tables(raw_file)

        for table_name, audit_df, encoding_name in loaded_tables:
            column_names = audit_df.columns.astype(str).tolist()
            numeric_columns, convertible_columns = identify_numeric_columns(audit_df)
            candidate_numeric_columns = numeric_columns + convertible_columns
            date_columns = find_columns_by_keywords(column_names, date_keywords)
            province_columns = find_columns_by_keywords(column_names, province_keywords)
            category_columns = find_columns_by_keywords(column_names, category_keywords)

            schema_records.append(
                {
                    "file_name": raw_file.name,
                    "table_name": table_name,
                    "encoding": encoding_name,
                    "row_count": audit_df.shape[0],
                    "column_count": audit_df.shape[1],
                    "duplicate_row_count": int(audit_df.duplicated().sum()),
                    "date_columns_found": ", ".join(date_columns),
                    "province_columns_found": ", ".join(province_columns),
                    "category_columns_found": ", ".join(category_columns),
                    "numeric_columns_found": ", ".join(numeric_columns),
                    "convertible_numeric_columns": ", ".join(convertible_columns),
                    "possible_analytics": summarize_possible_analytics(
                        date_columns,
                        province_columns,
                        category_columns,
                        candidate_numeric_columns,
                    ),
                }
            )

            for column_name in audit_df.columns:
                column_profile_records.append(
                    {
                        "file_name": raw_file.name,
                        "table_name": table_name,
                        "column_name": str(column_name),
                        "data_type": str(audit_df[column_name].dtype),
                        "missing_count": int(audit_df[column_name].isna().sum()),
                        "missing_rate": audit_df[column_name].isna().mean(),
                        "placeholder_missing_count": int(
                            audit_df[column_name].astype("string").str.strip().isin(["", "-"]).sum()
                        ),
                    }
                )
    except Exception as error:
        load_error_records.append(
            {
                "file_name": raw_file.name,
                "error_type": type(error).__name__,
                "error_message": str(error),
            }
        )

## แสดงสรุป schema

ตารางนี้ใช้ตรวจ column ที่พบจริงและวิธีวิเคราะห์ที่อาจเป็นไปได้ตามโครงสร้างข้อมูล

In [12]:
schema_audit_df = pd.DataFrame(schema_records)
schema_audit_df

,file_name,table_name,encoding,row_count,column_count,duplicate_row_count,date_columns_found,province_columns_found,category_columns_found,numeric_columns_found,convertible_numeric_columns,possible_analytics
0,labai_land_matching.xlsx,Sheet2,not_applicable,77,5,0,,จังหวัด,,,"จับคู่ได้, ผู้ประสงค์, เจ้าของที่ดิน, แปลงที่ดิน","descriptive summary, provincial comparison, nu..."
1,labai_registered_land_demand_supply.json,JSON,utf-8,77,11,0,,PROVINCE_NAME,,"countw, count_want_land, area_rai_want_land, a...",,"descriptive summary, provincial comparison, nu..."
2,labai_registered_landowners.json,JSON,utf-8,77,6,0,,PROVINCE_NAME,,"countowner, count_owner_land, area_rai, area_n...",,"descriptive summary, provincial comparison, nu..."
3,sacit_craft_community_2564_2568.xlsx,ชุมชนหัตถกรรม,not_applicable,51,6,0,ปี,จังหวัด,ชื่อชุมชนหัตถกรรม,"ลำดับ, ปี",,"descriptive summary, provincial comparison, ca..."
4,sacit_craft_product_overview_2568.csv,CSV,utf-8-sig,29,14,0,"ม.ค.68, ก.พ.68, มี.ค.68, เม.ย.68, พ.ค.68, มิ.ย...",,รายการสินค้า,"ลำดับ, ก.ย.68, ต.ค.68, พ.ย.68, ธ.ค.68","ม.ค.68, ก.พ.68, มี.ค.68, เม.ย.68, พ.ค.68, มิ.ย...","descriptive summary, category comparison, time..."


## แสดง data type และ missing values ราย column

ตารางนี้ช่วยระบุ column ที่ต้องตรวจ cleaning ใน Phase 3

In [13]:
column_profile_df = pd.DataFrame(column_profile_records)
column_profile_df

,file_name,table_name,column_name,data_type,missing_count,missing_rate,placeholder_missing_count
0,labai_land_matching.xlsx,Sheet2,จังหวัด,str,0,0.000000,0
1,labai_land_matching.xlsx,Sheet2,จับคู่ได้,str,0,0.000000,70
2,labai_land_matching.xlsx,Sheet2,ผู้ประสงค์,str,0,0.000000,70
3,labai_land_matching.xlsx,Sheet2,เจ้าของที่ดิน,str,0,0.000000,70
4,labai_land_matching.xlsx,Sheet2,แปลงที่ดิน,str,0,0.000000,70
5,labai_registered_land_demand_supply.json,JSON,PROVINCE_NAME,str,0,0.000000,0
6,labai_registered_land_demand_supply.json,JSON,countw,int64,0,0.000000,0
7,labai_registered_land_demand_supply.json,JSON,count_want_land,int64,0,0.000000,0
8,labai_registered_land_demand_supply.json,JSON,area_rai_want_land,int64,0,0.000000,0
9,labai_registered_land_demand_supply.json,JSON,area_nga_want_land,int64,0,0.000000,0


## แสดงข้อผิดพลาดในการอ่านไฟล์

หากตารางนี้ว่าง หมายความว่าไฟล์ที่รองรับทั้งหมดอ่านได้ในการรันครั้งนี้

In [14]:
load_errors_df = pd.DataFrame(load_error_records)
load_errors_df

""


## แหล่งข้อมูล

URL ของ metadata และ resource อยู่ใน `docs/data-sources.md`
การระบุ date, province, category และ numeric columns เป็นการตรวจจากชื่อและชนิดข้อมูลที่พบจริงเท่านั้น